# Class 4 · Food Delivery Data Exploration and Analysis 3

Data Foundations · AIML Jul-Aug26 Batch 3

Every example from the class, in order. Run a cell with **Shift+Enter**. Cells that use `input()` wait for you to type an answer below them.

- Notes for this class: the README.md file in the same folder.


## Sorting carry over

`00_sorting_carry_over.py`

Carry-over from Class 3: sorting.
np.sort gives the values (smallest first; [::-1] for largest first).
np.argsort gives the positions: use them on parallel arrays to keep each restaurant together.


In [ ]:
import numpy as np

votes = np.array([775, 787, 918, 88, 166,
                  286, 2556, 324, 504, 402])
costs = np.array([800., 800., 800., 300., 600.,
                  600., 600., 700., 550., 500.])

# The product team: "Sort by popularity", most votes first
print(np.sort(votes))
print(np.sort(votes)[::-1])
# np.sort(votes, reverse=True)
# TypeError: sort() got an unexpected keyword argument 'reverse'

# The manager: the three most popular places, and what they cost
print(np.argsort(votes))
order = np.argsort(votes)[::-1]
print(votes[order][:3])
print(costs[order][:3])

# The manager's sheet, ranked by votes: sort whole rows, not each column
data = np.column_stack((votes, costs))
print(np.sort(data, axis=0)[:3])            # each column sorted apart: rows broken
col = data[:, 0]          # 1. the votes column
order = np.argsort(col)   # 2. row numbers, fewest votes first
ranked = data[order]      # 3. whole rows, in that order
print(order)
print(ranked[:3])

# Numbers stored as text sort like words
prices = np.array(["800", "1200", "300", "550"])
print(np.sort(prices))


## Elementwise

`01_elementwise.py`

Element-wise maths: * + - / pair up the same positions of two arrays.
An array and one number: the number is used for every element.


In [ ]:
import numpy as np

# Our 10 practice restaurants: votes, rating out of 5, cost for two (Rs)
votes = np.array([775, 787, 918, 88, 166,
                  286, 2556, 324, 504, 402])
ratings = np.array([4.1, 4.1, 3.8, 3.7, 3.8,
                    3.8, 4.6, 4.0, 4.2, 4.1])
costs = np.array([800., 800., 800., 300., 600.,
                  600., 600., 700., 550., 500.])

# The manager: a 5.0 from 2 people or a 4.8 from 1000? A trust score uses both.
trust = votes * ratings
print(trust)
# Does the rating change the order? No: the votes (hundreds) swamp the ratings (about 4).
print(np.argsort(trust)[::-1])
print(np.argsort(votes)[::-1])

# Restaurants: every price goes up 5%
new_costs = costs * 1.05
print(new_costs)

# Shapes that do not fit give an error:
# votes * np.array([1, 2, 3, 4])
# ValueError: operands could not be broadcast together with shapes (10,) (4,)

a = np.array([2, 4, 6])
b = np.array([10, 20, 30])
print(a * b + 1)


## Matrix multiplication

`02_matrix_multiplication.py`

Matrix multiplication: a row times a column, multiplied in pairs, then added.
(A, B) @ (B, C) gives (A, C): the middle numbers must match.


In [ ]:
import numpy as np

# The app: one cart (2 biryani, 1 naan, 3 lassi) and the prices
prices = np.array([250, 40, 60])
cart = np.array([2, 1, 3])
print(cart * prices)     # each line of the bill
print(cart @ prices)     # the bill: 500 + 40 + 180 = 720

# Three customers checking out: one row per cart
carts = np.array([[2, 1, 3],
                  [1, 4, 0],
                  [0, 2, 2]])
print(carts @ prices)    # one bill per customer

# The same carts at two restaurants: one column per restaurant
price_lists = np.array([[250, 220],
                        [ 40,  50],
                        [ 60,  70]])
bills = carts @ price_lists
print(bills)             # row = customer, column = restaurant

# A price list for 4 dishes does not fit 3-dish carts:
# carts @ np.array([250, 40, 60, 30])
# ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0 ...

# @, np.dot and np.matmul agree on matrices
print(np.array_equal(bills, np.dot(carts, price_lists)),
      np.array_equal(bills, np.matmul(carts, price_lists)))

# Only np.dot accepts a single number (a @ 5 and np.matmul(a, 5) give an error)
a = np.array([1, 2, 3, 4])
print(np.dot(a, 5))

# * against @
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])
print(A * B)
print(A @ B)

# The ranking team: 70% rating, 30% popularity (votes in thousands)
votes = np.array([775, 787, 918, 88, 166, 286, 2556, 324, 504, 402])
ratings = np.array([4.1, 4.1, 3.8, 3.7, 3.8, 3.8, 4.6, 4.0, 4.2, 4.1])
features = np.array([ratings, votes / 1000]).T
weights = np.array([0.7, 0.3])
score = features @ weights
print(features[:3])
print(score.round(2))

# Why votes / 1000? Without it, the votes swamp the rating: the plain vote order again.
raw = np.array([ratings, votes]).T @ weights
print(np.argsort(raw)[::-1])
print(np.argsort(score)[::-1])


## Broadcasting

`03_broadcasting.py`

Broadcasting: a smaller array stretches to fit, when the shapes fit from the right
(equal, or one of them is 1). Nothing is copied.


In [ ]:
import numpy as np

# Start in 1D: one number, same length, a length that does not fit
prices = np.array([100, 200, 300])
print(prices + 20)                        # the 20 stretches to 3
print(prices + np.array([10, 20, 30]))    # same length: no stretch
print(prices + np.array([20]))            # length 1 stretches
# prices + np.array([10, 20])   ValueError: shapes (3,) (2,)

carts = np.array([[2, 1, 3], [1, 4, 0], [0, 2, 2]])
price_lists = np.array([[250, 220], [40, 50], [60, 70]])
bills = carts @ price_lists

# Pricing: Rs 30 delivery fee on every bill
print(bills + 30)

# Packaging per dish, added to the menus of two outlets: (2, 3) + (3,)
menu = np.array([[250, 40, 60],
                 [230, 45, 65]])
packing = np.array([10, 5, 15])
print(menu + packing)

# A sale preview: dishes at 100, 200, 400, each at 10, 20, 50% off. (3, 1) * (1, 3) gives (3, 3)
prices = np.array([[100], [200], [400]])
off = np.array([[0.9, 0.8, 0.5]])
print(prices.shape, off.shape)
print(prices * off)

# Shapes that fit, and one that does not
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a + np.array([[10], [20]]))      # (2, 3) and (2, 1): works
# a + np.array([[10, 20]])             # (2, 3) and (1, 2): ValueError

A = np.array([[1], [2], [3]])
b = np.array([10, 20, 30])
print(A + b)                           # (3, 1) and (3,): a 3 x 3 table


## Your turn

Add new cells below and write your own programs here.


In [ ]:
# your code
